# Parte 5 — Visualización con Plotly Express
### Recorrido guiado (con datos reales) sobre `clase_fundamentos_python_CD.md`, sección 5

La visualización es "la cereza del pastel": el mejor análisis pierde valor si no se comunica de forma clara. Plotly Express agrega interactividad (hover, zoom) casi gratis en comparación con matplotlib estático.

Se retoma el DataFrame `data` y `casos_pais` calculados en la Parte 4.

In [1]:
import pandas as pd
import plotly.express as px

data = pd.read_csv("covid_19_data.csv", parse_dates=["ObservationDate", "Last Update"])
casos_pais = data.groupby("Country/Region")["Confirmed"].max().reset_index()
casos_pais

,Country/Region,Confirmed
0,Australia,3835
1,France,4529
2,Mainland China,4259
3,Mexico,4365
4,US,4171


## 5.1 Gráfica de barras — Top de países por casos confirmados
Código literal de las notas de clase (`iloc[:20]` para quedarse con el top 20). Este dataset solo tiene 5 países, así que el `iloc[:20]` simplemente los toma a todos — no genera error, pero conviene saberlo antes de asumir que siempre habrá 20 barras.

In [2]:
fig = px.bar(
    casos_pais.sort_values("Confirmed", ascending=False).iloc[:20],
    x="Country/Region", y="Confirmed",
    title="Casos confirmados por país (Top 20)"
)
fig.show()

## 5.2 Gráfica de línea — Evolución de casos en EE.UU.
Código literal de las notas de clase.

In [3]:
fig = px.line(
    data[data["Country/Region"] == "US"],
    x="ObservationDate", y="Confirmed",
    hover_data=["Province/State"],
    title="Evolución de casos confirmados en EE.UU."
)
fig.show()

**Problema encontrado con datos reales:** esta gráfica *no* genera ningún error, pero el resultado es engañoso. `hover_data` solo agrega información al pasar el mouse — no separa ni agrupa las líneas. Como EE.UU. reporta por estado, para cada fecha hay varias filas (una por estado), y Plotly las conecta en el orden en que aparecen en el DataFrame, no por fecha ordenada ni por estado. El resultado es una línea que "zigzaguea" en vez de mostrar una tendencia clara. Se puede comprobar así:

In [4]:
us = data[data["Country/Region"] == "US"]
print("Filas para EE.UU.:", us.shape[0])
print("Estados distintos:", us["Province/State"].unique())
print("Fechas repetidas (una por estado):", us["ObservationDate"].duplicated().sum())

Filas para EE.UU.: 270
Estados distintos: <StringArray>
['New York', 'California', 'Texas']
Length: 3, dtype: str
Fechas repetidas (una por estado): 180


**Dos formas correctas de arreglarlo**, según lo que se quiera comunicar:

**Opción A — una línea por estado**, usando `color` para separarlas en vez de solo mandarlas como `hover_data`:

In [5]:
fig = px.line(
    us.sort_values("ObservationDate"),
    x="ObservationDate", y="Confirmed",
    color="Province/State",
    title="Evolución de casos confirmados en EE.UU. por estado"
)
fig.show()

**Opción B — una sola línea con el total nacional**, agregando (sumando) los estados por fecha antes de graficar — el mismo patrón de `groupby` + `reset_index` de la Parte 4:

In [6]:
us_total = us.groupby("ObservationDate")["Confirmed"].sum().reset_index()

fig = px.line(
    us_total,
    x="ObservationDate", y="Confirmed",
    title="Evolución de casos confirmados en EE.UU. (total nacional)"
)
fig.show()

---
### Resumen de la Parte 5
- `px.bar(df, x=..., y=..., title=...)` para comparar categorías (ej. países).
- `px.line(df, x=..., y=..., title=...)` para series de tiempo.
- `hover_data=[...]` solo agrega texto al pasar el mouse; **no** agrupa ni separa líneas.
- Si un filtro (como `"US"`) todavía tiene varias filas por fecha (por ejemplo, varios estados), hay que decidir explícitamente qué se quiere mostrar: **una línea por categoría** (`color="..."`) o **un total agregado** (`groupby(...).sum().reset_index()` antes de graficar). No hacerlo produce una gráfica que corre sin error pero comunica algo incorrecto — el mismo riesgo que menciona la sección 6.2 de las notas: "no asumir que los datos son correctos, explorar antes de transformar/graficar".